In [ ]:
#problme 2 - use autoencoder, use jax
#problme 3- take a sequence of the lfsr outputs and tehn we predict what the next point is, then shift by one and find the next output

In [40]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import itertools, random
import jax
import jax.numpy as jnp
import flax.linen as jax_nn
import optax

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)
key = jax.random.PRNGKey(SEED)

# Plotly white‑background template
import plotly.io as pio
pio.templates.default = "plotly_white"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cpu


# Question 1

In the PCA task, I used datasets from F1 to classify different tire degradation states and driver styles. For the NN, implementation, I then 

## CAn we predict the driving styles?

In [70]:
from __future__ import annotations

import sys
from pathlib import Path
from typing import Any
for _p in [Path('src'), Path('../src'), Path('../../src')]:
    _resolved = _p.resolve()
    if (_resolved / 'driver_style_windows.py').exists():
        if str(_resolved) not in sys.path:
            sys.path.insert(0, str(_resolved))
        break
import jax
import jax.numpy as jnp
import numpy as np
import optax
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split

from tire_stint_paths import default_tire_stint_features_csv

from driver_style_windows import GEO_KEYS, balance_to_min_per_class, build_windowed_xy
from grid_telemetry_paths import default_grid_telemetry_csv
print(f'JAX devices: {jax.devices()}')
LABEL_COL='driver_number'
DATE_COL='date'
CSV_PATH= default_grid_telemetry_csv()
WINDOW= 8      # consecutive samples per sequence (~2 s at ~4 Hz)
STRIDE= 2      # step between window starts along a segment
MAX_GAP_S= 3.0    # split segment if timestamps jump more than this (seconds)
INCLUDE_GEO= True   # append x,y,z,distance if present in the CSV
SEED= 0

JAX devices: [CpuDevice(id=0)]


In [56]:
x_raw, y_raw, class_labels, feat_cols = build_windowed_xy(
    CSV_PATH,
    label_col=LABEL_COL,
    date_col=DATE_COL,
    window=WINDOW,
    stride=STRIDE,
    max_gap_s=MAX_GAP_S,
    include_geo=INCLUDE_GEO,
)

n_classes = int(class_labels.size)
geo_used  = [c for c in feat_cols if c.lower() in GEO_KEYS]

print(f'Windows (raw)  : {x_raw.shape[0]:,}')
print(f'Feature dim    : {x_raw.shape[1]}  ({len(feat_cols)} channels x {WINDOW} steps)')
print(f'Drivers        : {n_classes}  -> {class_labels.tolist()}')
print(f'Channels       : {feat_cols}')
print(f'Geo channels   : {geo_used or "none"}')

Windows (raw)  : 179,060
Feature dim    : 40  (5 channels x 8 steps)
Drivers        : 20  -> [1, 2, 4, 10, 11, 14, 16, 18, 20, 22, 23, 24, 27, 31, 40, 44, 55, 63, 77, 81]
Channels       : ['speed', 'throttle', 'brake', 'rpm', 'n_gear']
Geo channels   : none


In [57]:
# Per-driver window counts before balancing
raw_counts = np.bincount(y_raw, minlength=n_classes)
print('Windows per driver (before balancing):')
for driver, cnt in zip(class_labels.tolist(), raw_counts.tolist()):
    print(f'  driver {driver:>3}: {cnt:,}')

Windows per driver (before balancing):
  driver   1: 8,953
  driver   2: 8,953
  driver   4: 8,953
  driver  10: 8,953
  driver  11: 8,953
  driver  14: 8,953
  driver  16: 8,953
  driver  18: 8,953
  driver  20: 8,953
  driver  22: 8,953
  driver  23: 8,953
  driver  24: 8,953
  driver  27: 8,953
  driver  31: 8,953
  driver  40: 8,953
  driver  44: 8,953
  driver  55: 8,953
  driver  63: 8,953
  driver  77: 8,953
  driver  81: 8,953


In [ ]:
BALANCE= True  # set False to keep all windows unbalanced
TRAIN_FRAC= 0.8

In [59]:
n_before = x_raw.shape[0]

if BALANCE:
    x_raw, y_raw, n_per_driver = balance_to_min_per_class(x_raw, y_raw, seed=SEED)
    print(
        f'Class-balanced: {n_before:,} -> {x_raw.shape[0]:,} windows '
        f'({n_per_driver} per driver x {n_classes} drivers)'
    )
else:
    counts = np.bincount(y_raw, minlength=n_classes)
    print(
        f'Balance skipped: {n_before:,} windows '
        f'(min={counts.min()}, max={counts.max()} per driver)'
    )

run_summary = (
    f'n_windows={x_raw.shape[0]}, x_dim={x_raw.shape[1]}, '
    f'window={WINDOW}, stride={STRIDE}, '
    f'balanced={BALANCE}, '
    f'channels={feat_cols}, geo={geo_used or "none"}'
)
print(run_summary)

Class-balanced: 179,060 -> 179,060 windows (8953 per driver x 20 drivers)
n_windows=179060, x_dim=40, window=8, stride=2, balanced=True, channels=['speed', 'throttle', 'brake', 'rpm', 'n_gear'], geo=none


In [60]:
def train_val_split(
    x: np.ndarray,
    y: np.ndarray,
    train_frac: float = 0.8,
    seed: int = 0,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    # Stratified so val row sums in the confusion matrix stay ~proportional
    _, counts = np.unique(y, return_counts=True)
    strat = y if int(np.min(counts)) >= 2 else None
    if strat is None:
        print('warning: stratify skipped (need >=2 samples per class)')
    x_tr, x_va, y_tr, y_va = train_test_split(
        x, y,
        train_size=train_frac,
        random_state=seed,
        shuffle=True,
        stratify=strat,
    )
    return np.asarray(x_tr), np.asarray(y_tr), np.asarray(x_va), np.asarray(y_va)


def standardize_x(
    x_train: np.ndarray,
    x_other: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    x_mu = x_train.mean(axis=0)
    x_sd = x_train.std(axis=0)
    x_sd = np.where(x_sd < 1e-8, 1.0, x_sd)
    return (x_train - x_mu) / x_sd, (x_other - x_mu) / x_sd

In [61]:
x_tr, y_tr, x_va, y_va = train_val_split(x_raw, y_raw, train_frac=TRAIN_FRAC, seed=SEED)
x_tr, x_va = standardize_x(x_tr, x_va)

vc = np.bincount(y_va, minlength=n_classes)
tc = np.bincount(y_tr, minlength=n_classes)
print(f'Train: {x_tr.shape[0]:,} windows   Val: {x_va.shape[0]:,} windows')
print(f'Val label counts  (confusion row sums)  — min={vc.min()} max={vc.max()}')
print(f'Train label counts                      — min={tc.min()} max={tc.max()}')

Train: 143,248 windows   Val: 35,812 windows
Val label counts  (confusion row sums)  — min=1790 max=1791
Train label counts                      — min=7162 max=7163


### Training the model

In [62]:
def init_mlp_params(
    key: jax.Array,
    in_dim: int,
    hidden: tuple[int, ...],
    out_dim: int,
) -> list[dict[str, jax.Array]]:
    layers: list[dict[str, jax.Array]] = []
    dims = (in_dim, *hidden, out_dim)
    keys = jax.random.split(key, len(dims) - 1)
    for i in range(len(dims) - 1):
        n_in, n_out = dims[i], dims[i + 1]
        scale = jnp.sqrt(2.0 / jnp.maximum(n_in, 1.0))
        w = jax.random.normal(keys[i], (n_in, n_out)) * scale
        b = jnp.zeros((n_out,))
        layers.append({'w': w, 'b': b})
    return layers


def forward_logits(
    params: list[dict[str, jax.Array]],
    x: jax.Array,
) -> jax.Array:
    h = x
    for i, layer in enumerate(params):
        h = h @ layer['w'] + layer['b']
        if i < len(params) - 1:
            h = jax.nn.relu(h)
    return h


def loss_ce(
    params: list[dict[str, jax.Array]],
    x: jax.Array,
    y: jax.Array,
) -> jax.Array:
    logits = forward_logits(params, x)
    return jnp.mean(
        optax.softmax_cross_entropy_with_integer_labels(logits=logits, labels=y)
    )

In [63]:
def make_train_step(tx: optax.GradientTransformation):
    @jax.jit
    def step(params, opt_state, xb, yb):
        loss, grads = jax.value_and_grad(loss_ce)(params, xb, yb)
        updates, opt_state = tx.update(grads, opt_state, params)
        return optax.apply_updates(params, updates), opt_state, loss
    return step


def iter_batches(
    x: np.ndarray,
    y: np.ndarray,
    batch_size: int,
    rng: np.random.Generator,
) -> list[tuple[np.ndarray, np.ndarray]]:
    n = x.shape[0]
    ix = rng.permutation(n)
    out: list[tuple[np.ndarray, np.ndarray]] = []
    for s in range(0, n, batch_size):
        j = ix[s:min(s + batch_size, n)]
        out.append((x[j], y[j]))
    return out


@jax.jit
def accuracy(
    params: list[dict[str, jax.Array]],
    x: jax.Array,
    y: jax.Array,
) -> jax.Array:
    pred = jnp.argmax(forward_logits(params, x), axis=-1)
    return jnp.mean((pred == y).astype(jnp.float32))

### I split out my hypterparameters so I could play around with them also

In [ ]:
HIDDEN      = (256, 128)
EPOCHS      = 300
BATCH_SIZE  = 256
LR          = 1e-3
PRINT_EVERY = 50

In [65]:
key = jax.random.PRNGKey(SEED)
key, k_init = jax.random.split(key)
params = init_mlp_params(k_init, x_tr.shape[1], HIDDEN, n_classes)

tx = optax.adam(LR)
opt_state = tx.init(params)
train_step = make_train_step(tx)

x_va_j = jnp.asarray(x_va, dtype=jnp.float32)
y_va_j = jnp.asarray(y_va, dtype=jnp.int32)
rng_batch = np.random.default_rng(SEED + 1)

train_losses: list[float] = []
val_losses: list[float]   = []
val_accs: list[float]     = []

for epoch in range(EPOCHS):
    ep_train: list[float] = []
    for xb, yb in iter_batches(x_tr, y_tr, BATCH_SIZE, rng_batch):
        xj = jnp.asarray(xb, dtype=jnp.float32)
        yj = jnp.asarray(yb, dtype=jnp.int32)
        params, opt_state, loss = train_step(params, opt_state, xj, yj)
        ep_train.append(float(loss))
    tl = float(np.mean(ep_train))
    vl = float(loss_ce(params, x_va_j, y_va_j))
    va = float(accuracy(params, x_va_j, y_va_j))
    train_losses.append(tl)
    val_losses.append(vl)
    val_accs.append(va)
    if (epoch + 1) % PRINT_EVERY == 0:
        print(f'epoch {epoch+1:4d}/{EPOCHS}  train_ce={tl:.4f}  val_ce={vl:.4f}  val_acc={va:.4f}')

print('Training complete.')

epoch   50/300  train_ce=1.9544  val_ce=2.0448  val_acc=0.3147
epoch  100/300  train_ce=1.8439  val_ce=2.0027  val_acc=0.3230
epoch  150/300  train_ce=1.7808  val_ce=1.9662  val_acc=0.3440
epoch  200/300  train_ce=1.7356  val_ce=1.9776  val_acc=0.3415
epoch  250/300  train_ce=1.7102  val_ce=1.9828  val_acc=0.3499
epoch  300/300  train_ce=1.6864  val_ce=1.9891  val_acc=0.3526
Training complete.


In [67]:
logits_va   = np.asarray(forward_logits(params, x_va_j))
pred_va     = np.argmax(logits_va, axis=1)
final_acc   = float(np.mean(pred_va == y_va))
pred_counts = np.bincount(pred_va, minlength=n_classes)

print(f'Drivers (classes)    : {n_classes}')
print(f'Val accuracy         : {final_acc:.4f}')
print(f'Chance baseline      : {1/n_classes:.4f}')
print(f'Val prediction counts (confusion column sums) — min={pred_counts.min()} max={pred_counts.max()}')

Drivers (classes)    : 20
Val accuracy         : 0.3526
Chance baseline      : 0.0500
Val prediction counts (confusion column sums) — min=659 max=7775


In [69]:
def _white(**kwargs: Any) -> dict[str, Any]:
    return {
        'template': 'plotly_white',
        'paper_bgcolor': '#ffffff',
        'plot_bgcolor': '#ffffff',
        **kwargs,
    }
k = class_labels.size
cm = np.zeros((k, k), dtype=np.int64)
for t, p in zip(y_va.tolist(), pred_va.tolist()):
    cm[int(t), int(p)] += 1

tick = [str(int(c)) for c in np.asarray(class_labels).ravel()]
cm_subtitle = (
    'row sums = val windows per true driver (flat if balanced). '
    'column sums = model prediction frequency — usually not flat.'
)

fig_cm = go.Figure(
    data=go.Heatmap(
        z=cm,
        x=tick,
        y=tick,
        colorscale='Blues',
        colorbar={'title': 'count'},
        text=cm,
        texttemplate='%{text}',
    )
)
fig_cm.update_layout(**_white(
    title=f'Validation confusion',
    xaxis_title='predicted driver',
    yaxis_title='actual driver',
))
fig_cm.show()

## Tire degradation evaluation - how good are the laps predicted?

In [73]:
ALL_FEATURE_COLS = [
    'Fuel_Corrected_Pace_Slope',
    'Out_Lap_Push_Factor',
    'Stint_Length_Normalized',
    'Throttle_Commitment_Delta',
    'Micro_Slip_Variance',
    'Braking_Distance_Extension',
    'Minimum_Apex_Speed_Delta',
    'Avg_Track_Temp',
    'Track_Temp_Volatility',
]
TARGET_COL = 'Fuel_Corrected_Pace_Slope'
INPUT_COLS = [c for c in ALL_FEATURE_COLS if c != TARGET_COL]
_ID_COLS = ['driver', 'event', 'stint', 'n_laps']

print(f'Target : {TARGET_COL}')
print(f'Inputs ({len(INPUT_COLS)}): {INPUT_COLS}')

Target : Fuel_Corrected_Pace_Slope
Inputs (8): ['Out_Lap_Push_Factor', 'Stint_Length_Normalized', 'Throttle_Commitment_Delta', 'Micro_Slip_Variance', 'Braking_Distance_Extension', 'Minimum_Apex_Speed_Delta', 'Avg_Track_Temp', 'Track_Temp_Volatility']


In [74]:
def load_xy(
    csv_path: Path,
    min_laps: int = 2,
) -> tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    df = pd.read_csv(csv_path)
    miss = [c for c in ALL_FEATURE_COLS if c not in df.columns]
    if miss:
        raise ValueError(f'missing columns: {miss}')
    miss_id = [c for c in _ID_COLS if c not in df.columns]
    if miss_id:
        raise ValueError(f'missing id columns: {miss_id}')

    dff = cast(pd.DataFrame, df[df['n_laps'] >= min_laps].copy())
    x = dff[INPUT_COLS].to_numpy(dtype=np.float64)
    y = dff[TARGET_COL].to_numpy(dtype=np.float64)
    m = np.isfinite(x).all(axis=1) & np.isfinite(y)
    dff = dff.loc[m].reset_index(drop=True)
    x, y = x[m], y[m]
    return x, y, dff


CSV_PATH = default_tire_stint_features_csv()
MIN_LAPS = 2

print(f'Loading: {CSV_PATH}')
x_raw, y_raw, df_clean = load_xy(CSV_PATH, min_laps=MIN_LAPS)

print(f'\nStints after filtering (>= {MIN_LAPS} laps): {len(df_clean)}')
print(f'Feature matrix shape: {x_raw.shape}')
df_clean[_ID_COLS + INPUT_COLS + [TARGET_COL]].head(10)

Loading: /Users/daryaguettler/NMM/outputs/tire_stint_features_2024.csv

Stints after filtering (>= 2 laps): 886
Feature matrix shape: (886, 8)


,driver,event,stint,n_laps,Out_Lap_Push_Factor,Stint_Length_Normalized,Throttle_Commitment_Delta,Micro_Slip_Variance,Braking_Distance_Extension,Minimum_Apex_Speed_Delta,Avg_Track_Temp,Track_Temp_Volatility,Fuel_Corrected_Pace_Slope
0,VER,Bahrain Grand Prix,1,13,-0.978600,1.173285,9.627407,1.900531,-296.498935,11.444444,23.533333,0.154560,-0.017615
1,VER,Bahrain Grand Prix,2,18,-1.109054,0.913580,-4.804815,9.535793,-15.424444,-2.666667,22.900000,0.210442,0.000810
2,VER,Bahrain Grand Prix,3,19,-4.666600,1.714801,8.954907,8.049087,11.180772,1.888889,22.086667,0.206128,-0.026468
3,PER,Bahrain Grand Prix,1,9,-0.733600,0.812274,13.089259,-16.539786,16.016728,13.777778,23.593750,0.129753,0.041433
4,PER,Bahrain Grand Prix,2,22,-0.786054,1.116598,10.338611,-15.988398,-19.556420,-11.111111,23.008571,0.257872,-0.041068
5,PER,Bahrain Grand Prix,3,20,-2.470600,1.805054,-16.151574,7.604124,232.704182,25.777778,22.103226,0.222136,0.006636
6,SAI,Bahrain Grand Prix,1,11,-0.038600,0.992780,16.200000,4.424785,-472.652500,34.555556,23.584211,0.122531,-0.024282
7,SAI,Bahrain Grand Prix,2,19,-0.016054,0.964335,5.641111,11.410442,20.959290,8.444444,22.983871,0.241095,-0.021693
8,SAI,Bahrain Grand Prix,3,21,-1.090054,1.065844,4.403981,6.007802,505.047006,7.555556,22.130303,0.240561,-0.035221
9,LEC,Bahrain Grand Prix,1,8,-0.412600,0.722022,-206.681111,-1.963424,-271.280849,8.222222,23.623077,0.124985,0.065119


### Building the test data

In [75]:
def train_val_split(
    x: np.ndarray,
    y: np.ndarray,
    train_frac: float = 0.8,
    seed: int = 0,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    n = x.shape[0]
    rng = np.random.default_rng(seed)
    idx = rng.permutation(n)
    n_train = max(1, int(n * train_frac))
    n_train = min(n_train, n - 1) if n > 1 else n
    tr, va = idx[:n_train], idx[n_train:]
    if va.size == 0:
        va = tr[-1:]
        tr = tr[:-1]
    return x[tr], y[tr], x[va], y[va]


def standardize(
    x_train: np.ndarray,
    x_other: np.ndarray,
    y_train: np.ndarray,
    y_other: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    x_mu = x_train.mean(axis=0)
    x_sd = x_train.std(axis=0)
    x_sd = np.where(x_sd < 1e-8, 1.0, x_sd)
    y_mu = float(y_train.mean())
    y_sd = float(y_train.std())
    if y_sd < 1e-8:
        y_sd = 1.0
    xt = (x_train - x_mu) / x_sd
    xo = (x_other - x_mu) / x_sd
    yt = (y_train - y_mu) / y_sd
    yo = (y_other - y_mu) / y_sd
    return xt, xo, yt, yo, x_mu, np.array([y_mu, y_sd])

In [76]:
TRAIN_FRAC = 0.8
SEED = 0

x_tr, y_tr, x_va, y_va = train_val_split(x_raw, y_raw, train_frac=TRAIN_FRAC, seed=SEED)
x_tr, x_va, y_tr_s, y_va_s, x_mu, y_stats = standardize(x_tr, x_va, y_tr, y_va)
y_mu, y_sd = float(y_stats[0]), float(y_stats[1])

print(f'Train: {x_tr.shape[0]} stints   Val: {x_va.shape[0]} stints')
print(f'Target  mean (train): {y_mu:.4f} s/lap   std: {y_sd:.4f} s/lap')

pd.DataFrame({
    'feature': INPUT_COLS,
    'train_mean': x_mu.round(4),
    'train_std': x_tr.std(axis=0).round(4),
})

Train: 708 stints   Val: 178 stints
Target  mean (train): -0.0545 s/lap   std: 0.1144 s/lap


,feature,train_mean,train_std
0,Out_Lap_Push_Factor,-0.0033,1.0
1,Stint_Length_Normalized,1.0512,1.0
2,Throttle_Commitment_Delta,-2.1441,1.0
3,Micro_Slip_Variance,0.8975,1.0
4,Braking_Distance_Extension,115.9416,1.0
5,Minimum_Apex_Speed_Delta,8.5880,1.0
6,Avg_Track_Temp,36.4353,1.0
7,Track_Temp_Volatility,0.7850,1.0


In [77]:
def init_mlp_params(
    key: jax.Array,
    in_dim: int,
    hidden: tuple[int, ...],
    out_dim: int = 1,
) -> list[dict[str, jax.Array]]:
    layers: list[dict[str, jax.Array]] = []
    dims = (in_dim,) + hidden + (out_dim,)
    keys = jax.random.split(key, len(dims) - 1)
    for i in range(len(dims) - 1):
        n_in, n_out = dims[i], dims[i + 1]
        scale = jnp.sqrt(2.0 / jnp.maximum(n_in, 1.0))
        w = jax.random.normal(keys[i], (n_in, n_out)) * scale
        b = jnp.zeros((n_out,))
        layers.append({'w': w, 'b': b})
    return layers


def forward(params: list[dict[str, jax.Array]], x: jax.Array) -> jax.Array:
    h = x
    for i, layer in enumerate(params):
        h = h @ layer['w'] + layer['b']
        if i < len(params) - 1:
            h = jax.nn.relu(h)
    return h.squeeze(-1)


def loss_mse(params: list[dict[str, jax.Array]], x: jax.Array, y: jax.Array) -> jax.Array:
    return jnp.mean((forward(params, x) - y) ** 2)

In [78]:
def make_train_step(tx: optax.GradientTransformation):
    @jax.jit
    def step(params, opt_state, xb, yb):
        loss, grads = jax.value_and_grad(loss_mse)(params, xb, yb)
        updates, opt_state = tx.update(grads, opt_state, params)
        return optax.apply_updates(params, updates), opt_state, loss
    return step


def iter_batches(
    x: np.ndarray,
    y: np.ndarray,
    batch_size: int,
    rng: np.random.Generator,
) -> list[tuple[np.ndarray, np.ndarray]]:
    n = x.shape[0]
    ix = rng.permutation(n)
    out: list[tuple[np.ndarray, np.ndarray]] = []
    for s in range(0, n, batch_size):
        j = ix[s:min(s + batch_size, n)]
        out.append((x[j], y[j]))
    return out

In [79]:
HIDDEN     = (64, 32)  # hidden layer sizes
EPOCHS     = 400
BATCH_SIZE = 32
LR         = 1e-3
PRINT_EVERY = 50       # print loss every N epochs

In [80]:
key = jax.random.PRNGKey(SEED)
key, k_init = jax.random.split(key)
params = init_mlp_params(k_init, x_tr.shape[1], HIDDEN)

tx = optax.adam(LR)
opt_state = tx.init(params)
train_step = make_train_step(tx)

x_va_j = jnp.asarray(x_va, dtype=jnp.float32)
y_va_j = jnp.asarray(y_va_s, dtype=jnp.float32)
rng_batch = np.random.default_rng(SEED + 1)

train_losses: list[float] = []
val_losses: list[float] = []

for epoch in range(EPOCHS):
    ep_train: list[float] = []
    for xb, yb in iter_batches(x_tr, y_tr_s, BATCH_SIZE, rng_batch):
        xj = jnp.asarray(xb, dtype=jnp.float32)
        yj = jnp.asarray(yb, dtype=jnp.float32)
        params, opt_state, loss = train_step(params, opt_state, xj, yj)
        ep_train.append(float(loss))
    tl = float(np.mean(ep_train))
    vl = float(loss_mse(params, x_va_j, y_va_j))
    train_losses.append(tl)
    val_losses.append(vl)
    if (epoch + 1) % PRINT_EVERY == 0:
        print(f'epoch {epoch+1:4d}/{EPOCHS}  train_mse={tl:.4f}  val_mse={vl:.4f}')

print('Training complete.')

epoch   50/400  train_mse=0.3781  val_mse=0.6816
epoch  100/400  train_mse=0.1973  val_mse=0.8902
epoch  150/400  train_mse=0.1264  val_mse=1.0755
epoch  200/400  train_mse=0.0969  val_mse=1.2620
epoch  250/400  train_mse=0.0700  val_mse=1.2402
epoch  300/400  train_mse=0.0543  val_mse=1.2936
epoch  350/400  train_mse=0.0659  val_mse=1.3574
epoch  400/400  train_mse=0.0345  val_mse=1.2829
Training complete.


In [81]:
pred_va_s = np.asarray(forward(params, x_va_j), dtype=np.float64)
pred_va   = pred_va_s * y_sd + y_mu

rmse = float(np.sqrt(np.mean((pred_va - y_va) ** 2)))
mae  = float(np.mean(np.abs(pred_va - y_va)))
print(f'Val RMSE : {rmse:.6f} s/lap')
print(f'Val MAE  : {mae:.6f} s/lap')

Val RMSE : 0.129523 s/lap
Val MAE  : 0.087219 s/lap


In [82]:
def _white(**kwargs: Any) -> dict[str, Any]:
    return {
        'template': 'plotly_white',
        'paper_bgcolor': '#ffffff',
        'plot_bgcolor': '#ffffff',
        **kwargs,
    }

lims = (float(min(y_va.min(), pred_va.min())), float(max(y_va.max(), pred_va.max())))
pad  = (lims[1] - lims[0]) * 0.05 + 1e-6
lo, hi = lims[0] - pad, lims[1] + pad

fig_pred = go.Figure()
fig_pred.add_trace(go.Scatter(
    x=y_va, y=pred_va, mode='markers', name='stints',
    marker={'size': 7, 'opacity': 0.55},
))
fig_pred.add_trace(go.Scatter(
    x=[lo, hi], y=[lo, hi], mode='lines', name='y=x',
    line={'dash': 'dash', 'color': 'gray'},
))
fig_pred.update_layout(**_white(
    title=f'Validation predictions  (RMSE={rmse:.4f}, MAE={mae:.4f})',
    xaxis_title='actual pace slope (s/lap)',
    yaxis_title='predicted pace slope (s/lap)',
))
fig_pred.show()

In [85]:
import plotly.io as pio
from IPython.display import HTML
display(HTML('outputs/tire_lstm/figures/tire_lstm_holdout_eval.html'))

# Question 2

In [86]:
import numpy as np
import itertools, random

# --- JAX ecosystem (Part 1) ---
import jax
import jax.numpy as jnp
import flax.linen as fnn
import optax

# --- PyTorch (Part 2) ---
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- Plotting ---
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates.default = "plotly_white"

# Reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
key = jax.random.PRNGKey(SEED)

print(f"JAX backend: {jax.default_backend()}")
print(f"JAX devices: {jax.devices()}")
print(f"PyTorch device: {device}")


JAX backend: cpu
JAX devices: [CpuDevice(id=0)]
PyTorch device: cpu


In [87]:
# DTMF frequency pairs
LOW_FREQS  = [697, 770, 852, 941]
HIGH_FREQS = [1209, 1336, 1477, 1633]
SYMBOLS    = ['1','2','3','A',
              '4','5','6','B',
              '7','8','9','C',
              '*','0','#','D']

SAMPLE_RATE = 8000   # Hz
DURATION    = 0.05   # 50 ms per tone  (400 samples)
N_SAMPLES_PER_SYMBOL = 500
SNR_DB_RANGE = (5, 20)  # additive white Gaussian noise

t = np.linspace(0, DURATION, int(SAMPLE_RATE * DURATION), endpoint=False).astype(np.float32)
N_TIME = len(t)
print(f"Samples per tone: {N_TIME}  |  Duration: {DURATION*1000:.0f} ms  |  Sample rate: {SAMPLE_RATE} Hz")

def generate_dtmf(low_freq, high_freq, snr_db):
    """Return a noisy DTMF waveform (1‑D numpy array)."""
    signal = np.sin(2 * np.pi * low_freq * t) + np.sin(2 * np.pi * high_freq * t)
    signal = signal / np.max(np.abs(signal))
    noise_power = 10 ** (-snr_db / 10)
    noise = np.sqrt(noise_power) * np.random.randn(N_TIME).astype(np.float32)
    return signal + noise

# Build dataset
X_all, y_all = [], []
for idx, (li, hi) in enumerate(itertools.product(range(4), range(4))):
    for _ in range(N_SAMPLES_PER_SYMBOL):
        snr = np.random.uniform(*SNR_DB_RANGE)
        waveform = generate_dtmf(LOW_FREQS[li], HIGH_FREQS[hi], snr)
        X_all.append(waveform)
        y_all.append(idx)

X_all = np.stack(X_all)
y_all = np.array(y_all)
print(f"Dataset shape: {X_all.shape}  |  Labels (for eval only): {len(np.unique(y_all))} classes")


Samples per tone: 400  |  Duration: 50 ms  |  Sample rate: 8000 Hz
Dataset shape: (8000, 400)  |  Labels (for eval only): 16 classes


## Using JAX/FLAX

In [88]:
class Encoder(fnn.Module):
    latent_dim: int = 8

    @fnn.compact
    def __call__(self, x, train: bool = True):
        # x: (B, T, 1) — channels last
        x = fnn.Conv(32, kernel_size=(15,), strides=(2,), padding='SAME')(x)   # -> (B, 200, 32)
        x = fnn.BatchNorm(use_running_average=not train)(x)
        x = fnn.relu(x)
        x = fnn.Conv(64, kernel_size=(7,), strides=(2,), padding='SAME')(x)    # -> (B, 100, 64)
        x = fnn.BatchNorm(use_running_average=not train)(x)
        x = fnn.relu(x)
        x = fnn.Conv(128, kernel_size=(5,), strides=(2,), padding='SAME')(x)   # -> (B, 50, 128)
        x = fnn.BatchNorm(use_running_average=not train)(x)
        x = fnn.relu(x)
        x = x.reshape(x.shape[0], -1)                                         # -> (B, 6400)
        x = fnn.Dense(self.latent_dim)(x)                                       # -> (B, latent_dim)
        return x


class Decoder(fnn.Module):
    output_len: int = 400

    @fnn.compact
    def __call__(self, z, train: bool = True):
        x = fnn.Dense(50 * 128)(z)                                              # -> (B, 6400)
        x = fnn.relu(x)
        x = x.reshape(-1, 50, 128)                                             # -> (B, 50, 128)
        x = fnn.ConvTranspose(64, kernel_size=(5,), strides=(2,), padding='SAME')(x)   # -> (B, 100, 64)
        x = fnn.BatchNorm(use_running_average=not train)(x)
        x = fnn.relu(x)
        x = fnn.ConvTranspose(32, kernel_size=(7,), strides=(2,), padding='SAME')(x)   # -> (B, 200, 32)
        x = fnn.BatchNorm(use_running_average=not train)(x)
        x = fnn.relu(x)
        x = fnn.ConvTranspose(1, kernel_size=(15,), strides=(2,), padding='SAME')(x)   # -> (B, 400, 1)
        x = x[:, :self.output_len, :]
        return x


class DTMFAutoencoder(fnn.Module):
    latent_dim: int = 8
    output_len: int = 400

    def setup(self):
        self.encoder = Encoder(latent_dim=self.latent_dim)
        self.decoder = Decoder(output_len=self.output_len)

    def __call__(self, x, train: bool = True):
        z = self.encoder(x, train=train)
        x_hat = self.decoder(z, train=train)
        return x_hat, z

    def encode(self, x, train: bool = False):
        return self.encoder(x, train=train)


# Initialise
LATENT_DIM = 8
model_ae = DTMFAutoencoder(latent_dim=LATENT_DIM, output_len=N_TIME)

dummy = jnp.ones((2, N_TIME, 1))
variables = model_ae.init(key, dummy, train=True)
params = variables['params']
batch_stats = variables['batch_stats']

n_params = sum(x.size for x in jax.tree.leaves(params))
print(f"Autoencoder initialised  |  Total parameters: {n_params:,}")

# Quick shape check
x_hat, z = model_ae.apply({'params': params, 'batch_stats': batch_stats},
                           dummy, train=False)
print(f"Forward pass:  input {dummy.shape} -> reconstruction {x_hat.shape}, latent {z.shape}")


Autoencoder initialised  |  Total parameters: 221,321
Forward pass:  input (2, 400, 1) -> reconstruction (2, 400, 1), latent (2, 8)


In [37]:
class Encoder(nn.Module):
    latent_dim: int = 8

    @nn.compact
    def __call__(self, x, train: bool = True):
        # x: (B, T, 1) — channels last
        x = nn.Conv(32, kernel_size=(15,), strides=(2,), padding='SAME')(x)   # -> (B, 200, 32)
        x = nn.BatchNorm(use_running_average=not train)(x)
        x = nn.relu(x)
        x = nn.Conv(64, kernel_size=(7,), strides=(2,), padding='SAME')(x)    # -> (B, 100, 64)
        x = nn.BatchNorm(use_running_average=not train)(x)
        x = nn.relu(x)
        x = nn.Conv(128, kernel_size=(5,), strides=(2,), padding='SAME')(x)   # -> (B, 50, 128)
        x = nn.BatchNorm(use_running_average=not train)(x)
        x = nn.relu(x)
        x = x.reshape(x.shape[0], -1)                                         # -> (B, 6400)
        x = nn.Dense(self.latent_dim)(x)                                       # -> (B, latent_dim)
        return x


class Decoder(nn.Module):
    output_len: int = 400

    @nn.compact
    def __call__(self, z, train: bool = True):
        x = nn.Dense(50 * 128)(z)                                              # -> (B, 6400)
        x = nn.relu(x)
        x = x.reshape(-1, 50, 128)                                             # -> (B, 50, 128)
        x = nn.ConvTranspose(64, kernel_size=(5,), strides=(2,), padding='SAME')(x)   # -> (B, 100, 64)
        x = nn.BatchNorm(use_running_average=not train)(x)
        x = nn.relu(x)
        x = nn.ConvTranspose(32, kernel_size=(7,), strides=(2,), padding='SAME')(x)   # -> (B, 200, 32)
        x = nn.BatchNorm(use_running_average=not train)(x)
        x = nn.relu(x)
        x = nn.ConvTranspose(1, kernel_size=(15,), strides=(2,), padding='SAME')(x)   # -> (B, 400, 1)
        x = x[:, :self.output_len, :]
        return x


class DTMFAutoencoder(nn.Module):
    latent_dim: int = 8
    output_len: int = 400
    def setup(self):
        self.encoder = Encoder(latent_dim=self.latent_dim)
        self.decoder = Decoder(output_len=self.output_len)
    def __call__(self, x, train: bool = True):
        z = self.encoder(x, train=train)
        x_hat = self.decoder(z, train=train)
        return x_hat, z
    def encode(self, x, train: bool = False):
        return self.encoder(x, train=train)


# Initialise
LATENT_DIM = 8
model_ae = DTMFAutoencoder(latent_dim=LATENT_DIM, output_len=N_TIME)

dummy = jnp.ones((2, N_TIME, 1))
variables = model_ae.init(key, dummy, train=True)
params = variables['params']
batch_stats = variables['batch_stats']

n_params = sum(x.size for x in jax.tree.leaves(params))
print(f"Autoencoder initialised  |  Total parameters: {n_params:,}")

# Quick shape check
x_hat, z = model_ae.apply({'params': params, 'batch_stats': batch_stats},
                           dummy, train=False)
print(f"Forward pass:  input {dummy.shape} -> reconstruction {x_hat.shape}, latent {z.shape}")


Autoencoder initialised  |  Total parameters: 221,321
Forward pass:  input (2, 400, 1) -> reconstruction (2, 400, 1), latent (2, 8)


In [89]:
# --- Prepare data as JAX arrays ---
X_jax = jnp.array(X_all[:, :, None])  # (N, T, 1)

BATCH_SIZE = 256
EPOCHS = 60
N = len(X_all)

# Optimiser with cosine decay
schedule = optax.cosine_decay_schedule(init_value=1e-3, decay_steps=EPOCHS * (N // BATCH_SIZE + 1))
tx = optax.adam(learning_rate=schedule)
opt_state = tx.init(params)

# --- JIT‑compiled training step ---
@jax.jit
def train_step(params, batch_stats, opt_state, batch):
    def loss_fn(params):
        (x_hat, z), updates = model_ae.apply(
            {'params': params, 'batch_stats': batch_stats},
            batch, train=True, mutable=['batch_stats']
        )
        loss = jnp.mean((x_hat - batch) ** 2)
        return loss, updates

    (loss, updates), grads = jax.value_and_grad(loss_fn, has_aux=True)(params)
    param_updates, opt_state_new = tx.update(grads, opt_state, params)
    params_new = optax.apply_updates(params, param_updates)
    return params_new, updates['batch_stats'], opt_state_new, loss

# --- Training loop ---
rng = np.random.default_rng(SEED)
losses = []

for epoch in range(1, EPOCHS + 1):
    perm = rng.permutation(N)
    epoch_loss = 0.0
    n_batches = 0
    for start in range(0, N, BATCH_SIZE):
        idx = perm[start : start + BATCH_SIZE]
        batch = X_jax[idx]
        params, batch_stats, opt_state, loss = train_step(params, batch_stats, opt_state, batch)
        epoch_loss += float(loss)
        n_batches += 1
    avg = epoch_loss / n_batches
    losses.append(avg)
    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{EPOCHS}  loss={avg:.6f}")

fig = go.Figure(go.Scatter(y=losses, mode='lines', name='MSE Loss'))
fig.update_layout(title="Autoencoder Training Loss",
                  xaxis_title="Epoch", yaxis_title="MSE",
                  template="plotly_white", width=700, height=350)
fig.show()


Epoch   1/60  loss=0.276364
Epoch  10/60  loss=0.089236
Epoch  20/60  loss=0.089044
Epoch  30/60  loss=0.088012
Epoch  40/60  loss=0.088073
Epoch  50/60  loss=0.087261
Epoch  60/60  loss=0.087756


In [90]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from scipy.optimize import linear_sum_assignment
from collections import Counter
Z = np.array(model_ae.apply(
    {'params': params, 'batch_stats': batch_stats},
    X_jax, train=False, method=model_ae.encode
))

kmeans = KMeans(n_clusters=16, n_init=20, random_state=SEED)
pred_clusters = kmeans.fit_predict(Z)

# ---- Hungarian‑method accuracy ----
def cluster_accuracy(y_true, y_pred, n_classes=16):
    cost = np.zeros((n_classes, n_classes))
    for c in range(n_classes):
        mask = y_pred == c
        counts = Counter(y_true[mask])
        for lbl, cnt in counts.items():
            cost[c, lbl] = -cnt
    row_ind, col_ind = linear_sum_assignment(cost)
    mapping = dict(zip(row_ind, col_ind))
    correct = sum((mapping[c] == y_true[i]) for i, c in enumerate(y_pred))
    return correct / len(y_true), mapping

acc, mapping = cluster_accuracy(y_all, pred_clusters)
ari = adjusted_rand_score(y_all, pred_clusters)
nmi = normalized_mutual_info_score(y_all, pred_clusters)

print(f"Clustering Accuracy (Hungarian): {acc:.2%}")
print(f"Adjusted Rand Index:             {ari:.4f}")
print(f"Normalised Mutual Information:   {nmi:.4f}")


Clustering Accuracy (Hungarian): 100.00%
Adjusted Rand Index:             1.0000
Normalised Mutual Information:   1.0000


/Users/daryaguettler/NMM/.venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:227: RuntimeWarning:

divide by zero encountered in matmul

/Users/daryaguettler/NMM/.venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:227: RuntimeWarning:

overflow encountered in matmul

/Users/daryaguettler/NMM/.venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:227: RuntimeWarning:

invalid value encountered in matmul

/Users/daryaguettler/NMM/.venv/lib/python3.12/site-packages/sklearn/cluster/_kmeans.py:243: RuntimeWarning:

divide by zero encountered in matmul

/Users/daryaguettler/NMM/.venv/lib/python3.12/site-packages/sklearn/cluster/_kmeans.py:243: RuntimeWarning:

overflow encountered in matmul

/Users/daryaguettler/NMM/.venv/lib/python3.12/site-packages/sklearn/cluster/_kmeans.py:243: RuntimeWarning:

invalid value encountered in matmul



In [91]:
fig = make_subplots(rows=4, cols=4, subplot_titles=SYMBOLS,
                    vertical_spacing=0.06, horizontal_spacing=0.04)

for idx in range(16):
    r, c = divmod(idx, 4)
    sample = X_jax[y_all == idx][0:1]  # (1, T, 1)
    recon, _ = model_ae.apply(
        {'params': params, 'batch_stats': batch_stats},
        sample, train=False
    )
    orig = np.array(sample.squeeze())
    rec  = np.array(recon.squeeze())
    fig.add_trace(go.Scatter(x=t*1000, y=orig, mode='lines', name='Original',
                             line=dict(width=0.8, color='steelblue'),
                             showlegend=(idx==0)),
                  row=r+1, col=c+1)
    fig.add_trace(go.Scatter(x=t*1000, y=rec, mode='lines', name='Reconstructed',
                             line=dict(width=0.8, color='coral', dash='dot'),
                             showlegend=(idx==0)),
                  row=r+1, col=c+1)

fig.update_layout(height=700, width=900,
                  title_text="Autoencoder Reconstructions vs Originals",
                  template="plotly_white")
fig.show()


### Also tried with torch to see the difference

In [4]:
class DTMFAutoencoder(nn.Module):
    """1‑D convolutional autoencoder with a compact latent space."""
    def __init__(self, latent_dim=8):
        super().__init__()
        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=15, stride=2, padding=7),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Conv1d(64, 128, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(4),         # -> (B, 128, 4)
            nn.Flatten(),                     # -> (B, 512)
            nn.Linear(512, latent_dim),
        )
        # Decoder
        self.decoder_fc = nn.Linear(latent_dim, 512)
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(128, 64, kernel_size=5, stride=2, padding=2, output_padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.ConvTranspose1d(64, 32, kernel_size=7, stride=2, padding=3, output_padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.ConvTranspose1d(32, 16, kernel_size=7, stride=2, padding=3, output_padding=1),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.ConvTranspose1d(16, 1, kernel_size=15, stride=2, padding=7, output_padding=1),
        )

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        h = self.decoder_fc(z).view(-1, 128, 4)
        out = self.decoder(h)
        return out

    def forward(self, x):
        z = self.encode(x)
        x_hat = self.decode(z)
        # Trim / pad to match input length
        if x_hat.size(-1) > x.size(-1):
            x_hat = x_hat[..., :x.size(-1)]
        elif x_hat.size(-1) < x.size(-1):
            x_hat = nn.functional.pad(x_hat, (0, x.size(-1) - x_hat.size(-1)))
        return x_hat, z

LATENT_DIM = 8
model_ae = DTMFAutoencoder(latent_dim=LATENT_DIM).to(device)
print(model_ae)
print(f"\nTotal parameters: {sum(p.numel() for p in model_ae.parameters()):,}")


DTMFAutoencoder(
  (encoder): Sequential(
    (0): Conv1d(1, 32, kernel_size=(15,), stride=(2,), padding=(7,))
    (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv1d(32, 64, kernel_size=(7,), stride=(2,), padding=(3,))
    (4): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): Conv1d(64, 128, kernel_size=(5,), stride=(2,), padding=(2,))
    (7): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (8): ReLU()
    (9): AdaptiveAvgPool1d(output_size=4)
    (10): Flatten(start_dim=1, end_dim=-1)
    (11): Linear(in_features=512, out_features=8, bias=True)
  )
  (decoder_fc): Linear(in_features=8, out_features=512, bias=True)
  (decoder): Sequential(
    (0): ConvTranspose1d(128, 64, kernel_size=(5,), stride=(2,), padding=(2,), output_padding=(1,))
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
   

In [5]:
# Prepare DataLoader — no labels used!
X_tensor = torch.tensor(X_all, dtype=torch.float32).unsqueeze(1)  # (N, 1, T)
dataset  = TensorDataset(X_tensor)
loader   = DataLoader(dataset, batch_size=256, shuffle=True)

optimizer = optim.Adam(model_ae.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=60)
criterion = nn.MSELoss()

EPOCHS = 60
losses = []

model_ae.train()
for epoch in range(1, EPOCHS + 1):
    epoch_loss = 0.0
    for (batch_x,) in loader:
        batch_x = batch_x.to(device)
        x_hat, _ = model_ae(batch_x)
        loss = criterion(x_hat, batch_x)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * batch_x.size(0)
    scheduler.step()
    avg = epoch_loss / len(dataset)
    losses.append(avg)
    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{EPOCHS}  loss={avg:.6f}")

fig = go.Figure(go.Scatter(y=losses, mode='lines', name='MSE Loss'))
fig.update_layout(title="Autoencoder Training Loss",
                  xaxis_title="Epoch", yaxis_title="MSE",
                  template="plotly_white", width=700, height=350)
fig.show()


Epoch   1/60  loss=0.345885
Epoch  10/60  loss=0.301136
Epoch  20/60  loss=0.300524
Epoch  30/60  loss=0.300211
Epoch  40/60  loss=0.299904
Epoch  50/60  loss=0.299676
Epoch  60/60  loss=0.299602


In [6]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from scipy.optimize import linear_sum_assignment

model_ae.eval()
with torch.no_grad():
    Z = model_ae.encode(X_tensor.to(device)).cpu().numpy()

kmeans = KMeans(n_clusters=16, n_init=20, random_state=SEED)
pred_clusters = kmeans.fit_predict(Z)

# ---- Hungarian‑method accuracy ----
def cluster_accuracy(y_true, y_pred, n_classes=16):
    """Best 1‑to‑1 mapping between clusters and true labels (Hungarian)."""
    from collections import Counter
    cost = np.zeros((n_classes, n_classes))
    for c in range(n_classes):
        mask = y_pred == c
        counts = Counter(y_true[mask])
        for lbl, cnt in counts.items():
            cost[c, lbl] = -cnt  # negative because we minimise
    row_ind, col_ind = linear_sum_assignment(cost)
    mapping = dict(zip(row_ind, col_ind))
    correct = sum((mapping[c] == y_true[i]) for i, c in enumerate(y_pred))
    return correct / len(y_true), mapping

acc, mapping = cluster_accuracy(y_all, pred_clusters)
ari = adjusted_rand_score(y_all, pred_clusters)
nmi = normalized_mutual_info_score(y_all, pred_clusters)

print(f"Clustering Accuracy (Hungarian): {acc:.2%}")
print(f"Adjusted Rand Index:             {ari:.4f}")
print(f"Normalised Mutual Information:   {nmi:.4f}")


/Users/daryaguettler/NMM/.venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:227: RuntimeWarning:

divide by zero encountered in matmul

/Users/daryaguettler/NMM/.venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:227: RuntimeWarning:

overflow encountered in matmul

/Users/daryaguettler/NMM/.venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:227: RuntimeWarning:

invalid value encountered in matmul

/Users/daryaguettler/NMM/.venv/lib/python3.12/site-packages/sklearn/cluster/_kmeans.py:243: RuntimeWarning:

divide by zero encountered in matmul

/Users/daryaguettler/NMM/.venv/lib/python3.12/site-packages/sklearn/cluster/_kmeans.py:243: RuntimeWarning:

overflow encountered in matmul

/Users/daryaguettler/NMM/.venv/lib/python3.12/site-packages/sklearn/cluster/_kmeans.py:243: RuntimeWarning:

invalid value encountered in matmul



Clustering Accuracy (Hungarian): 99.98%
Adjusted Rand Index:             0.9995
Normalised Mutual Information:   0.9993


In [52]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, perplexity=40, random_state=SEED, max_iter=1000)
Z2 = tsne.fit_transform(Z)

label_names = [SYMBOLS[i] for i in y_all]

fig = px.scatter(x=Z2[:, 0], y=Z2[:, 1],
                 color=label_names,
                 title="t‑SNE of Autoencoder Latent Space",
                 labels={'x': 't‑SNE 1', 'y': 't‑SNE 2', 'color': 'Symbol'},
                 opacity=0.5, width=800, height=600,
                 template="plotly_white")
fig.update_traces(marker=dict(size=4))
fig.show()


/Users/daryaguettler/NMM/.venv/lib/python3.12/site-packages/sklearn/decomposition/_base.py:152: RuntimeWarning:

divide by zero encountered in matmul

/Users/daryaguettler/NMM/.venv/lib/python3.12/site-packages/sklearn/decomposition/_base.py:152: RuntimeWarning:

overflow encountered in matmul

/Users/daryaguettler/NMM/.venv/lib/python3.12/site-packages/sklearn/decomposition/_base.py:152: RuntimeWarning:

invalid value encountered in matmul



In [10]:
fig = make_subplots(rows=4, cols=4, subplot_titles=SYMBOLS)

model_ae.eval()
with torch.no_grad():
    for idx in range(16):
        r, c = divmod(idx, 4)
        sample = X_tensor[y_all == idx][0:1].to(device)
        recon, _ = model_ae(sample)
        orig  = sample.cpu().squeeze().numpy()
        rec   = recon.cpu().squeeze().numpy()
        fig.add_trace(go.Scatter(x=t*1000, y=orig, mode='lines', name='Original',
                                 line=dict(width=0.8, color='steelblue'),
                                 showlegend=(idx==0)),
                      row=r+1, col=c+1)
        fig.add_trace(go.Scatter(x=t*1000, y=rec, mode='lines', name='Reconstructed',
                                 line=dict(width=0.8, color='coral', dash='dot'),
                                 showlegend=(idx==0)),
                      row=r+1, col=c+1)

fig.update_layout(height=700, width=900,
                  title_text="Autoencoder Reconstructions vs Originals",
                  template="plotly_white")
fig.show()


# Question 3

In [11]:
def lfsr(seed, taps, n_bits):
    """
    Galois LFSR.
    
    Parameters
    ----------
    seed : int   – initial register value (non‑zero)
    taps : list  – bit positions that are XOR‑ed (0‑indexed from LSB)
    n_bits : int – register width
    
    Returns
    -------
    list of int (0/1) – output bit sequence (one full period)
    """
    state = seed
    period = (1 << n_bits) - 1  # maximal‑length LFSR
    sequence = []
    for _ in range(period):
        out_bit = state & 1
        sequence.append(out_bit)
        feedback = 0
        for tap in taps:
            feedback ^= (state >> tap) & 1
        state = (state >> 1) | (feedback << (n_bits - 1))
    return sequence

# 8‑bit LFSR with primitive polynomial x^8 + x^6 + x^5 + x^4 + 1
# Taps at positions 0, 4, 5, 6 (from the characteristic polynomial)
LFSR_BITS = 8
LFSR_TAPS = [0, 4, 5, 6]
LFSR_SEED = 0b10110011

seq = lfsr(LFSR_SEED, LFSR_TAPS, LFSR_BITS)
print(f"LFSR register width: {LFSR_BITS}")
print(f"Sequence length (one period): {len(seq)}  (expected 2^{LFSR_BITS}−1 = {2**LFSR_BITS - 1})")
print(f"First 64 bits: {''.join(map(str, seq[:64]))}")

# Visualise autocorrelation
seq_np = np.array(seq, dtype=np.float32)
seq_centered = seq_np - seq_np.mean()
acorr = np.correlate(seq_centered, seq_centered, mode='full')
acorr = acorr[len(acorr)//2:]  # keep positive lags
acorr = acorr / acorr[0]

fig = go.Figure(go.Scatter(y=acorr[:200], mode='lines', line=dict(width=1)))
fig.update_layout(title="LFSR Output — Autocorrelation (first 200 lags)",
                  xaxis_title="Lag", yaxis_title="Normalised Autocorrelation",
                  template="plotly_white", width=700, height=350)
fig.show()


LFSR register width: 8
Sequence length (one period): 255  (expected 2^8−1 = 255)
First 64 bits: 1100110111011100101010010100010010110100011001110011110001101100


In [12]:
# Tile the sequence so the network sees multiple periods
full_seq = np.array(seq * 6, dtype=np.float32)

# We use 16 to give the network some margin.
WINDOW = 16

X_lfsr, y_lfsr = [], []
for i in range(len(full_seq) - WINDOW):
    X_lfsr.append(full_seq[i : i + WINDOW])
    y_lfsr.append(full_seq[i + WINDOW])

X_lfsr = np.stack(X_lfsr)
y_lfsr = np.array(y_lfsr)

#set my test/train split - I just withheld the last sequence
split = len(seq) * 5
X_train, X_test = X_lfsr[:split], X_lfsr[split:]
y_train, y_test = y_lfsr[:split], y_lfsr[split:]


Window: 16 bits  |  Total samples: 1514
Training samples: 1275  |  Test samples: 239


In [13]:
class LFSRPredictorMLP(nn.Module):
    def __init__(self, window_size, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(window_size, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


class LFSRPredictorLSTM(nn.Module):
    """LSTM baseline that reads bits one at a time."""
    def __init__(self, hidden_size=64, n_layers=2):
        super().__init__()
        self.lstm=nn.LSTM(input_size=1, hidden_size=hidden_size,
                            num_layers=n_layers, batch_first=True)
        self.head=nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        # x: (B, T, 1)
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :]).squeeze(-1)
model_mlp  = LFSRPredictorMLP(window_size=WINDOW).to(device)
model_lstm = LFSRPredictorLSTM(hidden_size=64, n_layers=2).to(device)

=== MLP ===
LFSRPredictorMLP(
  (net): Sequential(
    (0): Linear(in_features=16, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=1, bias=True)
  )
)
Parameters: 10,497

=== LSTM (baseline) ===
LFSRPredictorLSTM(
  (lstm): LSTM(1, 64, num_layers=2, batch_first=True)
  (head): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=1, bias=True)
  )
)
Parameters: 52,545


In [14]:
#making dataset
X_train_t=torch.tensor(X_train, dtype=torch.float32)
y_train_t=torch.tensor(y_train, dtype=torch.float32)
X_test_t=torch.tensor(X_test,  dtype=torch.float32)
y_test_t=torch.tensor(y_test,  dtype=torch.float32)

train_ds=TensorDataset(X_train_t, y_train_t)
test_ds=TensorDataset(X_test_t,  y_test_t)
train_loader=DataLoader(train_ds, batch_size=256, shuffle=True)
test_loader=DataLoader(test_ds,  batch_size=512)

#set my loss
criterion = nn.BCEWithLogitsLoss()
#I picked 100 epochs (somewhat arbitrary, but seemed reasonable for convergence)
EPOCHS = 100
def train_model(model, name, needs_unsqueeze=False):
    """Train a model and return loss/accuracy history."""
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=40, gamma=0.3)
    train_losses, test_accs = [], []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        epoch_loss=0.0
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            if needs_unsqueeze:
                bx = bx.unsqueeze(-1)
            logits = model(bx)
            loss = criterion(logits, by)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * bx.size(0)
        scheduler.step()
        avg_loss = epoch_loss / len(train_ds)
        train_losses.append(avg_loss)
        model.eval()
        correct = 0
        with torch.no_grad():
            for bx, by in test_loader:
                bx, by = bx.to(device), by.to(device)
                if needs_unsqueeze:
                    bx = bx.unsqueeze(-1)
                preds = (torch.sigmoid(model(bx)) > 0.5).float()
                correct += (preds == by).sum().item()
        acc = correct / len(test_ds)
        test_accs.append(acc)
        if epoch % 20 == 0 or epoch == 1:
            print(f"  [{name}] Epoch {epoch:3d}/{EPOCHS}  loss={avg_loss:.5f}  test_acc={acc:.2%}")
    return train_losses, test_accs

#mlp training
mlp_losses, mlp_accs = train_model(model_mlp, "MLP", needs_unsqueeze=False)

#lstm training
lstm_losses, lstm_accs = train_model(model_lstm, "LSTM", needs_unsqueeze=True)


Training MLP...
  [MLP] Epoch   1/100  loss=0.69469  test_acc=54.39%
  [MLP] Epoch  20/100  loss=0.54250  test_acc=97.91%
  [MLP] Epoch  40/100  loss=0.09016  test_acc=100.00%
  [MLP] Epoch  60/100  loss=0.04771  test_acc=100.00%
  [MLP] Epoch  80/100  loss=0.02861  test_acc=100.00%
  [MLP] Epoch 100/100  loss=0.02455  test_acc=100.00%

Training LSTM (baseline)...
  [LSTM] Epoch   1/100  loss=0.69450  test_acc=49.37%
  [LSTM] Epoch  20/100  loss=0.69315  test_acc=49.37%
  [LSTM] Epoch  40/100  loss=0.69314  test_acc=49.37%
  [LSTM] Epoch  60/100  loss=0.69311  test_acc=48.12%
  [LSTM] Epoch  80/100  loss=0.69310  test_acc=47.70%
  [LSTM] Epoch 100/100  loss=0.69309  test_acc=48.12%


In [31]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["Training Loss", "Test Accuracy"])

fig.add_trace(go.Scatter(y=mlp_losses, mode='lines', name='MLP Loss',
                         line=dict(color='steelblue')), row=1, col=1)
fig.add_trace(go.Scatter(y=lstm_losses, mode='lines', name='LSTM Loss',
                         line=dict(color='coral')), row=1, col=1)

fig.add_trace(go.Scatter(y=mlp_accs, mode='lines', name='MLP Acc',
                         line=dict(color='steelblue')), row=1, col=2)
fig.add_trace(go.Scatter(y=lstm_accs, mode='lines', name='LSTM Acc',
                         line=dict(color='coral')), row=1, col=2)

fig.update_layout(height=400, width=850, template="plotly_white",
                  title_text="compoare mlp and lsstm")
fig.update_xaxes(title_text="Epoch")
fig.update_yaxes(title_text="BCE Loss", row=1, col=1)
fig.update_yaxes(title_text="Accuracy", row=1, col=2, range=[0, 1.05])
fig.show()


In [30]:
GEN_LENGTH=300
seed_start=50
seed_window=np.array(seq[seed_start : seed_start + WINDOW], dtype=np.float32)
extended_seq = np.array(seq * 10, dtype=np.float32)
true_continuation = extended_seq[seed_start + WINDOW : seed_start + WINDOW + GEN_LENGTH]
model_mlp.eval()
generated = []
window = seed_window.copy()
with torch.no_grad():
    for _ in range(GEN_LENGTH):
        inp = torch.tensor(window, dtype=torch.float32).unsqueeze(0).to(device)  # (1, WINDOW)
        logit = model_mlp(inp)
        bit = 1.0 if torch.sigmoid(logit).item() > 0.5 else 0.0
        generated.append(bit)
        window = np.concatenate([window[1:], np.array([bit], dtype=np.float32)])
generated = np.array(generated)
match = (generated == true_continuation[:GEN_LENGTH]).astype(float)
run_acc = np.cumsum(match) / (np.arange(GEN_LENGTH) + 1)

print(f"accuracy: {match.mean():.2%}")
first_err = np.where(match == 0)[0]


accuracy: 100.00%


In [28]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=["True LFSR Sequence",
                                    "Network‑Generated Sequence"],vertical_spacing=0.08)

fig.add_trace(go.Scatter(y=true_continuation[:GEN_LENGTH],name='True'),
              row=1, col=1)
fig.add_trace(go.Scatter(y=generated, name='Generated'),
              row=2, col=1)

fig.update_layout(height=550, width=850, template="plotly_white",
                  title_text="checking accuracy")
fig.update_xaxes(title_text="Bit index", row=3)
fig.show()
